# <div align = "center"> Entrenamiento aereo </div>

<div align="justify"> Considere que una de las bases de la Fuerza Aérea debe <strong>planificar los ejercicios de vuelo</strong> de los distintos escuadrones que componen la escuela de aviación. Para las <strong>próximas semanas</strong> se debe decidir <strong><u>cuántas horas de vuelo se le asignaría a cada escuadrón</u></strong>, y cómo realizar el abastecimiento de combustible, principal insumo para realizar de los ejercicios.</div>

<div align="justify">Se sabe que durante todo el periodo que se está evaluando cada escuadrón debe realizar en total exactamente <strong>H horas de vuelo</strong>, y que en cada semana cada escuadrón debe realizar como <strong>mínimo Mi y como máximo Ni horas</strong> de ejercicios de vuelo. </div>

<div align="justify">Los ejercicios de vuelo que debe hacer cada escuadrón son distintos, <strong>por cada hora de vuelo del escuadrón i se consumirán ai litros de combustible.</strong> Por acuerdos con las distintas distribuidoras de combustible del país, <strong>comprar un litro de combustible comprado en la semana t tiene un costo de $ct.</strong> No existe un límite máximo a la cantidad de combustible que se puede comprar.</div>

<div align="justify">Al finalizar cada semana, la base tiene capacidad para <strong>reservar hasta K litros de combustible para la semana siguiente</strong>, a un costo de <strong>$v por litro almacenado</strong>, y que, por razones de seguridad, <strong>siempre se debe mantener una reserva de combustible en inventario de como mínimo P litros.</strong> Asuma que al inicio de la planificación existen L litros de combustible por sobre la reserva mínima como inventario inicial.</div>

<div align = "justify"> En base a este enunciado y el archivo excel con los datos formule y resuelva un modelo de optimización lineal que permita planificar los ejercicios de vuelo y el abastecimiento de combustible a mínimo costo. </div>

Datos : 
* P = 1000
* K = 1000000
* L = 200000
* v = 58000

___
### Librerías

In [23]:
import pandas as pd
import gurobipy as gp

### Importar datos

In [24]:
Escuadrones = pd.read_excel("Plantilla_Planificacion_Fuerza_Aerea.xlsx", sheet_name = "Escuadrones")
Semanas = pd.read_excel("Plantilla_Planificacion_Fuerza_Aerea.xlsx", sheet_name = "Semanas")
Minimo = pd.read_excel("Plantilla_Planificacion_Fuerza_Aerea.xlsx", sheet_name = "Minimo_semana_escuadron")
Maximo = pd.read_excel("Plantilla_Planificacion_Fuerza_Aerea.xlsx", sheet_name = "Maximo_semana_escuadron")

### Conjuntos

In [25]:
E = Escuadrones["Escuadron"].tolist()
S = Semanas["Semana"].tolist()

### Parámetros

In [ ]:
Consumo = Escuadrones["Consumo por hora (a_i)"].tolist()
Horas_total = Escuadrones["Horas totales"].tolist() 
minimo = Minimo["Horas minimas"].tolist()
maximo = Maximo["Horas maximas"].tolist()
costos = Semanas['Costo por litro (c_t)'].tolist()

P = 1000
K = 1000000
L = 200000
v = 58000

### El modelo

In [27]:
m = gp.Model("Costos de ejercicios de vuelo")

x_e_s = m.addVars(((e,s) for e in E for s in S), vtype= gp.GRB.CONTINUOUS, lb = 0, name ='Horas de vuelo')
y_s = m.addVars((s for s in S), vtype= gp.GRB.CONTINUOUS, lb = 0, name ='combustible comprado')
z_s = m.addVars((s for s in [0]+S), vtype= gp.GRB.CONTINUOUS, lb = 0, name ='combustible guardado en semana s para s+1') # [0]+S para poder establecer un inventario inicial


### Función objetivo

In [28]:
m.setObjective(gp.quicksum(costos[s-1]*y_s[s] + v*z_s[s] for s in S), gp.GRB.MINIMIZE)

### Restricciones:

> Los que aparecen con indice "algo - 1" son porque los parámetros están en listas.
> 
> Recomiendo hacer un **diccionario** para que sea más fácil de indexar y no tener que estar calculando el índice.

In [29]:
#R1: Cantidades maxima y mínima de horas de vuelo por escuadrón en cada semana,
m.addConstrs(x_e_s[e, s] >= minimo[s - 1] for e in E for s in S if s > 0)
m.addConstrs(x_e_s[e, s] <= maximo[s - 1] for e in E for s in S if s > 0)

#R2: Cantidad total de horas de vuelo que debe recibir cada escuadrón
m.addConstrs(gp.quicksum(x_e_s[e, s] for s in S if s > 0) == Horas_total[e - 1] for e in E)

#R3: Relación inter-temporal entre decisiones
m.addConstr(z_s[0] == P + L) # Inventario inicial (lo que se guardó en 0 para la semana 1)
m.addConstrs(z_s[s] == z_s[s - 1] + y_s[s] - gp.quicksum(Consumo[e - 1] * x_e_s[e, s] for e in E) for s in S)

#R4: Reservas mínimas de combustible y capacidad máxima de almacenamiento
m.addConstrs(z_s[s] >= P for s in S)
m.addConstrs(z_s[s] <= K  for s in S)

{1: <gurobi.Constr *Awaiting Model Update*>,
 2: <gurobi.Constr *Awaiting Model Update*>,
 3: <gurobi.Constr *Awaiting Model Update*>,
 4: <gurobi.Constr *Awaiting Model Update*>,
 5: <gurobi.Constr *Awaiting Model Update*>,
 6: <gurobi.Constr *Awaiting Model Update*>,
 7: <gurobi.Constr *Awaiting Model Update*>,
 8: <gurobi.Constr *Awaiting Model Update*>,
 9: <gurobi.Constr *Awaiting Model Update*>,
 10: <gurobi.Constr *Awaiting Model Update*>,
 11: <gurobi.Constr *Awaiting Model Update*>,
 12: <gurobi.Constr *Awaiting Model Update*>,
 13: <gurobi.Constr *Awaiting Model Update*>,
 14: <gurobi.Constr *Awaiting Model Update*>,
 15: <gurobi.Constr *Awaiting Model Update*>,
 16: <gurobi.Constr *Awaiting Model Update*>,
 17: <gurobi.Constr *Awaiting Model Update*>,
 18: <gurobi.Constr *Awaiting Model Update*>,
 19: <gurobi.Constr *Awaiting Model Update*>,
 20: <gurobi.Constr *Awaiting Model Update*>}

### Optimizamos

In [30]:
m.optimize()

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 266 rows, 141 columns and 501 nonzeros
Model fingerprint: 0x6a6cab72
Coefficient statistics:
  Matrix range     [1e+00, 3e+03]
  Objective range  [7e+02, 6e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+06]
Presolve removed 261 rows and 56 columns
Presolve time: 0.00s
Presolved: 5 rows, 85 columns, 85 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.5178208e+09   2.080000e+02   0.000000e+00      0s
       5    2.5849234e+09   0.000000e+00   0.000000e+00      0s

Solved in 5 iterations and 0.00 seconds (0.00 work units)
Optimal objective  2.584923360e+09


### Valores de variables

In [33]:
print(f'inventario inicial {z_s[0].X} [Lts.]')
print("\n----------------------------------------------")
for s in S:
    print(f"Semana {s}:\n")
    print(f"* litros comprados {y_s[s].X}")
    print(f"* litros almacenados para la semana {s+1}: {z_s[s].X}\n")
    print("Horas por escuadron:\n")
    for e in E:
        print(f'* Escuadron {e} = {x_e_s[(e, s)].X} hrs')
    print("\n----------------------------------------------")

print(f"Costo total del periodo de entrenamiento: ${m.ObjVal}")

inventario inicial 21000.0 [Lts.]

----------------------------------------------
Semana 1:

* litros comprados 79797.70000000001
* litros almacenados para la semana 2: 1000.0

Horas por escuadron:

* Escuadron 1 = 1.7 hrs
* Escuadron 2 = 10.0 hrs
* Escuadron 3 = 10.0 hrs
* Escuadron 4 = 1.7 hrs
* Escuadron 5 = 10.0 hrs

----------------------------------------------
Semana 2:

* litros comprados 21037.8
* litros almacenados para la semana 3: 1000.0

Horas por escuadron:

* Escuadron 1 = 1.4 hrs
* Escuadron 2 = 1.4 hrs
* Escuadron 3 = 1.4 hrs
* Escuadron 4 = 1.4 hrs
* Escuadron 5 = 1.4 hrs

----------------------------------------------
Semana 3:

* litros comprados 79968.6
* litros almacenados para la semana 4: 1000.0

Horas por escuadron:

* Escuadron 1 = 2.2 hrs
* Escuadron 2 = 10.0 hrs
* Escuadron 3 = 2.2 hrs
* Escuadron 4 = 2.2 hrs
* Escuadron 5 = 10.0 hrs

----------------------------------------------
Semana 4:

* litros comprados 59024.399999999994
* litros almacenados para la 